# Capítulo 3 — Pandas I: Series, DataFrames, carga y selección

**Serie Ciencia de Datos · Tomo 2 — Herramientas Python para Data Science**  
José Fernando Meza

Código del capítulo, extraído del libro. Repositorio: https://github.com/josefer76/serie-ciencia-de-datos

> Ejecute las celdas en orden: algunas dependen de variables definidas antes.

## Código del capítulo

### Introducción del capítulo

In [ ]:
!pip install -q pandas==2.3.3

### 3.1 Del diccionario de listas al DataFrame: cierre del puente pedagógico

In [ ]:
# Estructura nativa del Libro 1: diccionario de listas
registro_ventas = {
    "producto": ["Teclado", "Monitor", "Mouse", "Monitor"],
    "unidades": [12, 5, 30, 8],
    "precio_unitario": [25.0, 180.0, 12.5, 180.0],
}

# Para calcular el ingreso de cada venta, hacía falta un bucle explícito
# que recorriera las listas en paralelo por posición:
ingresos_manual = []
for indice in range(len(registro_ventas["producto"])):
    ingresos_manual.append(
        registro_ventas["unidades"][indice]
        * registro_ventas["precio_unitario"][indice]
    )
print(ingresos_manual)   # [300.0, 900.0, 375.0, 1440.0]

In [ ]:
import pandas as pd

# el dict de listas se convierte directamente
ventas = pd.DataFrame(registro_ventas)
# vectorizado, sin bucle
ventas["ingreso"] = ventas["unidades"] * ventas["precio_unitario"]
print(ventas)
#   producto  unidades  precio_unitario  ingreso
# 0  Teclado        12             25.0    300.0
# 1  Monitor         5            180.0    900.0
# 2    Mouse        30             12.5    375.0
# 3  Monitor         8            180.0   1440.0

### 3.2 La Series: array etiquetado; el índice como estructura de primera clase

In [ ]:
import pandas as pd

serie = pd.Series([25.0, 180.0, 12.5], index=["Teclado", "Monitor", "Mouse"])
print(serie)
# Teclado     25.0
# Monitor    180.0
# Mouse       12.5
# dtype: float64

In [ ]:
print(serie["Monitor"])    # 180.0  -> acceso por ETIQUETA
print(serie.iloc[1])       # 180.0  -> acceso por POSICIÓN

In [ ]:
print(type(serie.values))
# <class 'numpy.ndarray'>  -> NumPy sigue estando debajo
print(serie * 1.16)         # operación vectorizada, conservando las etiquetas
# Teclado     29.0
# Monitor    208.8
# Mouse       14.5
# dtype: float64

In [ ]:
serie_a = pd.Series([1, 2, 3], index=["a", "b", "c"])
# ¡índice en orden inverso!
serie_b = pd.Series([10, 20, 30], index=["c", "b", "a"])

# 'a' se suma con 'a': 1 (en serie_a) + 30 (en serie_b) = 31
print(serie_a + serie_b)
# a    31
# b    22
# c    13
# dtype: int64

### 3.3 El DataFrame: construcción desde dicts, listas de dicts y arrays

In [ ]:
import pandas as pd
import numpy as np

# 1. Desde un diccionario de listas (o de Series): cada clave -> una columna.
#    Es la forma más frecuente y la que cerró el puente pedagógico en 3.1.
desde_dict = pd.DataFrame({"col_a": [1, 2], "col_b": [3, 4]})

# 2. Desde una lista de diccionarios: cada dict -> una FILA.
#    Es la forma natural cuando los datos llegan de una API en formato JSON.
desde_registros = pd.DataFrame([
    {"col_a": 1, "col_b": 3},
    {"col_a": 2, "col_b": 4},
])

# 3. Desde un ndarray de NumPy, especificando los nombres de columna.
#    Útil cuando se viene de un cálculo numérico y hay que etiquetar el
#    resultado.
desde_array = pd.DataFrame(np.arange(6).reshape(3, 2), columns=["x", "y"])

### 3.4 Lectura de datos reales: `read_csv()` y sus parámetros críticos

In [ ]:
from pathlib import Path

Path("ventas_sucio.csv").write_text(
    "producto;unidades;precio;fecha_venta\n"
    "Teclado;12;25,0;2024-01-15\n"
    "Monitor;N/D;180,0;15/02/2024\n"
    "Mouse;30;12,5;2024-03-01\n",
    encoding="utf-8",
)
print(Path("ventas_sucio.csv").read_text(encoding="utf-8"))

*Salida en el libro:*

```text
producto;unidades;precio;fecha_venta
Teclado;12;25,0;2024-01-15
Monitor;N/D;180,0;15/02/2024
Mouse;30;12,5;2024-03-01
```

In [ ]:
import pandas as pd

crudo = pd.read_csv("ventas_sucio.csv")
print(crudo.shape)      # (3, 1)  -> ¡una sola columna!
print(crudo.columns)
# Index(['producto;unidades;precio;fecha_venta'], dtype='object')

In [ ]:
limpio = pd.read_csv(
    "ventas_sucio.csv",
    # separador de campos real del archivo
    sep=";",
    # coma decimal (convención hispanohablante)
    decimal=",",
    # codificación; 'latin-1' en exports antiguos de Windows
    encoding="utf-8",
    # cadenas que deben leerse como faltante (NaN)
    na_values=["N/D", "sin dato", "-"],
)
print(limpio)
#   producto  unidades  precio fecha_venta
# 0  Teclado      12.0    25.0  2024-01-15
# 1  Monitor       NaN   180.0  15/02/2024
# 2    Mouse      30.0    12.5  2024-03-01

In [ ]:
Path("codigos.csv").write_text("codigo\n00123\n00456\n", encoding="utf-8")

# Sin dtype: '00123' se lee como el entero 123 y los ceros iniciales se
# pierden.
print(pd.read_csv("codigos.csv")["codigo"].tolist())
# [123, 456]
print(pd.read_csv("codigos.csv", dtype={"codigo": str})["codigo"].tolist())
# ['00123', '00456']

In [ ]:
Path("ventas.csv").write_text(
    "producto,unidades,fecha_venta\n"
    "Teclado,12,2024-01-15\n"
    "Monitor,5,2024-02-15\n"
    "Mouse,30,2024-03-01\n",
    encoding="utf-8",
)
fechas = pd.read_csv("ventas.csv", parse_dates=["fecha_venta"])
print(fechas.dtypes)
# producto               object
# unidades                int64
# fecha_venta    datetime64[ns]
# dtype: object

In [ ]:
# 03/04/2024 es, en convención hispanohablante, el 3 de abril.
Path("fechas.csv").write_text("fecha\n03/04/2024\n05/06/2024\n",
                              encoding="utf-8")

sin_aviso = pd.read_csv("fechas.csv", parse_dates=["fecha"])
print(sin_aviso["fecha"].iloc[0])     # 2024-03-04 00:00:00 -> ¡4 de marzo!
con_dayfirst = pd.read_csv("fechas.csv", parse_dates=["fecha"], dayfirst=True)
print(con_dayfirst["fecha"].iloc[0])  # 2024-04-03 00:00:00 -> correcto

In [ ]:
datos = pd.read_csv("fechas.csv")   # se carga sin parsear fechas
# patrón declarado
datos["fecha"] = pd.to_datetime(datos["fecha"], format="%d/%m/%Y")
print(datos["fecha"].dtype)         # datetime64[ns]
print(datos["fecha"].iloc[0])       # 2024-04-03 00:00:00 (3 de abril)

In [ ]:
# dos formatos en una columna
mezcla = pd.Series(["03/04/2024", "2024-05-06"])
try:
    pd.to_datetime(mezcla, format="%d/%m/%Y")
except ValueError as error:
    print(f"ValueError: {error}")

*Salida en el libro:*

```text
ValueError: time data "2024-05-06" doesn't match format "%d/%m/%Y", at position 1. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.
```

### 3.5 Reconocimiento inicial: `head()`, `info()`, `describe()`, `shape`, `dtypes`, `value_counts()`

In [ ]:
import pandas as pd

datos = pd.DataFrame({
    "producto": ["Teclado", "Monitor", "Mouse", "Monitor", "Teclado",
                 "Mouse"],
    "unidades": [12, 5, 30, 8, None, 25],
    "precio": [25.0, 180.0, 12.5, 180.0, 25.0, 12.5],
})

In [ ]:
# Primeras 5 filas (head(n) para otra cantidad): ¿tiene la pinta esperada?
print(datos.head())
# (filas, columnas): ¿se cargó todo el archivo, o solo una parte?
print(datos.shape)
# Tipo de cada columna: ¿los números son números y las fechas, fechas?
print(datos.dtypes)
# Resumen: tipos, conteo de NO nulos por columna y uso de memoria
datos.info()
# Estadísticos descriptivos de las columnas numéricas
print(datos.describe())
# Frecuencia de cada valor de una columna categórica
print(datos["producto"].value_counts())

### 3.6 Selección rigurosa: `.loc` (por etiqueta) frente a `.iloc` (por posición)

In [ ]:
import pandas as pd

df = pd.DataFrame(
    {"unidades": [12, 5, 30], "precio": [25.0, 180.0, 12.5]},
    index=["Teclado", "Monitor", "Mouse"],
)

print(df.loc["Monitor", "precio"])
# 180.0  -> fila etiquetada 'Monitor', columna 'precio'
print(df.iloc[1, 1])
# 180.0  -> fila en posición 1, columna en posición 1

In [ ]:
print(df.loc["Teclado":"Monitor"])   # INCLUYE 'Monitor' (el extremo final)
#          unidades  precio
# Teclado        12    25.0
# Monitor         5   180.0

print(df.iloc[0:2])                  # EXCLUYE la posición 2 (como en NumPy)
#          unidades  precio
# Teclado        12    25.0
# Monitor         5   180.0

In [ ]:
# índice entero, desordenado
df_num = pd.DataFrame({"v": [10, 20, 30]}, index=[2, 0, 1])
print(df_num.loc[0, "v"])
# 20  -> la fila cuya ETIQUETA es 0 (la segunda fila)
print(df_num.iloc[0, 0])     # 10  -> la fila en POSICIÓN 0 (la primera fila)

### 3.7 Filtrado booleano, filtros compuestos y el `SettingWithCopyWarning`

In [ ]:
import pandas as pd

df = pd.DataFrame(
    {"unidades": [12, 5, 30], "precio": [25.0, 180.0, 12.5]},
    index=["Teclado", "Monitor", "Mouse"],
)
# Filas donde la condición es verdadera
print(df[df["unidades"] > 10])
#          unidades  precio
# Teclado        12    25.0
# Mouse          30    12.5

# Filtro compuesto
print(df[(df["unidades"] > 10) & (df["precio"] < 100)])
#          unidades  precio
# Teclado        12    25.0
# Mouse          30    12.5

#### El `SettingWithCopyWarning` explicado a fondo

In [ ]:
import pandas as pd

datos = pd.DataFrame({"cat": ["a", "b", "a"], "valor": [1, 2, 3]})

# PRIMERA indexación -> objeto intermedio
subconjunto = datos[datos["cat"] == "a"]
# SEGUNDA indexación sobre ese intermedio
subconjunto["valor"] = 99

*Salida en el libro:*

```text
SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subconjunto["valor"] = 99
```

In [ ]:
# el DataFrame original quedó INTACTO: la asignación se perdió
print(datos)
#   cat  valor
# 0   a      1
# 1   b      2
# 2   a      3

In [ ]:
# UNA sola indexación: filas y columna a la vez
datos.loc[datos["cat"] == "a", "valor"] = 99
print(datos)
#   cat  valor
# 0   a     99
# 1   b      2
# 2   a     99

In [ ]:
# copia explícita e intencional
subconjunto = datos[datos["cat"] == "a"].copy()
# modificarla ya no advierte ni sorprende
subconjunto["valor"] = 99

#### Lo que cambia en pandas 3

In [ ]:
import warnings

import pandas as pd

# Activa en pandas 2.x el comportamiento que pandas 3 trae por defecto
pd.options.mode.copy_on_write = True

datos = pd.DataFrame({"cat": ["a", "b", "a"], "valor": [1, 2, 3]})

# Caso 1: dos instrucciones. El subconjunto se comporta como una copia.
subconjunto = datos[datos["cat"] == "a"]
subconjunto["valor"] = 99
print("subconjunto:", subconjunto["valor"].tolist())
print("original   :", datos["valor"].tolist())

# Caso 2: asignación encadenada en una sola instrucción. Nunca funciona.
with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    datos["valor"][datos["cat"] == "a"] = 99
print("original   :", datos["valor"].tolist())
print("aviso      :", avisos[0].category.__name__)

# La forma correcta es la misma en pandas 2 y en pandas 3
datos.loc[datos["cat"] == "a", "valor"] = 99
print("con .loc   :", datos["valor"].tolist())

# Se restaura el comportamiento de pandas 2 para el resto del capítulo
pd.options.mode.copy_on_write = False

*Salida en el libro:*

```text
subconjunto: [99, 99]
original   : [1, 2, 3]
original   : [1, 2, 3]
aviso      : ChainedAssignmentError
con .loc   : [99, 2, 99]
```

### 3.8 Escritura y persistencia: `to_csv()`, `to_excel()`, `to_parquet()`

In [ ]:
import pandas as pd

ventas = pd.DataFrame({
    "producto": ["Teclado", "Monitor", "Mouse", "Monitor"],
    "unidades": [12, 5, 30, 8],
    "precio_unitario": [25.0, 180.0, 12.5, 180.0],
})

# index=False evita una columna basura
ventas.to_csv("ventas.csv", index=False, encoding="utf-8")
# requiere openpyxl instalado
ventas.to_excel("ventas.xlsx", index=False)
# requiere pyarrow instalado
ventas.to_parquet("ventas.parquet", index=False)

### 3.9 Optimización de memoria en la carga: downcasting numérico y categóricas

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
n = 100_000
grande = pd.DataFrame({
    "id_transaccion": np.arange(n, dtype=np.int64),
    "cantidad": rng.integers(1, 100, size=n),
    "precio": rng.random(n) * 500,
    "region": rng.choice(["Norte", "Sur", "Este", "Oeste"], size=n),
    "categoria": rng.choice(["Electrónica", "Hogar", "Ropa"], size=n),
})

memoria_antes = grande.memory_usage(deep=True).sum()
print(f"Memoria ANTES: {memoria_antes / 1024**2:.2f} MB")

*Salida en el libro:*

```text
Memoria ANTES: 13.18 MB
```

In [ ]:
optimizado = grande.copy()

# 'unsigned' para columnas sin valores negativos; 'integer' si pueden ser
# negativas.
optimizado["id_transaccion"] = pd.to_numeric(optimizado["id_transaccion"],
                                             downcast="unsigned")
optimizado["cantidad"] = pd.to_numeric(optimizado["cantidad"],
                                       downcast="unsigned")
optimizado["precio"] = pd.to_numeric(optimizado["precio"], downcast="float")

In [ ]:
print(np.iinfo(np.uint8).max)     # 255          -> suficiente para 'cantidad'
print(np.iinfo(np.uint32).max)
# 4294967295   -> suficiente para 'id_transaccion'

In [ ]:
# rango conocido: 1 a 99
optimizado["cantidad"] = optimizado["cantidad"].astype("uint8")

In [ ]:
# el valor 300 NO cabe en uint8 (máximo 255)
serie = pd.Series([10, 300, 50])

print(pd.to_numeric(serie, downcast="unsigned").tolist())
# [10, 300, 50]  -> eligió uint16, dato intacto
print(serie.astype("uint8").tolist())
# [10, 44, 50]  -> ¡300 se convirtió en 44!

In [ ]:
optimizado["region"] = optimizado["region"].astype("category")
optimizado["categoria"] = optimizado["categoria"].astype("category")

memoria_despues = optimizado.memory_usage(deep=True).sum()
print(f"Memoria DESPUÉS: {memoria_despues / 1024**2:.2f} MB")
print(f"Ahorro: {(1 - memoria_despues / memoria_antes) * 100:.1f}%")

*Salida en el libro:*

```text
Memoria DESPUÉS: 1.05 MB
Ahorro: 92.0%
```

In [ ]:
print(grande["precio"].iloc[0])
# 441.0980242229144   -> float64, ~15-17 dígitos significativos
print(optimizado["precio"].iloc[0])
# 441.09802           -> float32, ~7 dígitos significativos

#### El texto en pandas 3: un punto de partida distinto, la misma conclusión

In [ ]:
import numpy as np
import pandas as pd

# Activa en pandas 2.3 el tipo de texto que pandas 3 trae por defecto
pd.options.future.infer_string = True

rng = np.random.default_rng(42)
n = 100_000
grande = pd.DataFrame({
    "id_transaccion": np.arange(n, dtype=np.int64),
    "cantidad": rng.integers(1, 100, size=n),
    "precio": rng.random(n) * 500,
    "region": rng.choice(["Norte", "Sur", "Este", "Oeste"], size=n),
    "categoria": rng.choice(["Electrónica", "Hogar", "Ropa"], size=n),
})
print(grande.dtypes)

memoria_antes = grande.memory_usage(deep=True).sum()
print(f"Memoria ANTES: {memoria_antes / 1024**2:.2f} MB")

optimizado = grande.copy()
optimizado["id_transaccion"] = pd.to_numeric(optimizado["id_transaccion"],
                                             downcast="unsigned")
optimizado["cantidad"] = pd.to_numeric(optimizado["cantidad"],
                                       downcast="unsigned")
optimizado["precio"] = pd.to_numeric(optimizado["precio"], downcast="float")
optimizado["region"] = optimizado["region"].astype("category")
optimizado["categoria"] = optimizado["categoria"].astype("category")

memoria_despues = optimizado.memory_usage(deep=True).sum()
print(f"Memoria DESPUÉS: {memoria_despues / 1024**2:.2f} MB")
print(f"Ahorro: {(1 - memoria_despues / memoria_antes) * 100:.1f}%")

# Se restaura el tipo de texto de pandas 2 para el resto del capítulo
pd.options.future.infer_string = False

*Salida en el libro:*

```text
id_transaccion      int64
cantidad            int64
precio            float64
region                str
categoria             str
dtype: object
Memoria ANTES: 4.89 MB
Memoria DESPUÉS: 1.05 MB
Ahorro: 78.5%
```

In [ ]:
# 100 000 filas de ejemplo
grande.to_csv("archivo_grande.csv", index=False)

total_filas = 0
for fragmento in pd.read_csv("archivo_grande.csv", chunksize=50_000):
    total_filas += len(fragmento)   # se procesa fragmento a fragmento
print(total_filas)   # 100000

## Soluciones de los ejercicios

### [Ex-L2-C3-01] Del diccionario de listas al DataFrame

#### Desarrollo con código

In [ ]:
import pandas as pd

registro_ventas = {
    "producto": ["Teclado", "Monitor", "Mouse", "Webcam", "Monitor",
                 "Teclado"],
    "unidades": [12, 5, 30, 8, 3, 20],
    "precio_unitario": [25.0, 180.0, 12.5, 45.0, 180.0, 25.0],
    "region": ["Norte", "Sur", "Norte", "Este", "Sur", "Este"],
}

# ---------------------------------------------
# VERSIÓN NATIVA (la del Libro 1)
# ---------------------------------------------

# 1. Total agregado
ingreso_total_nativo = 0.0
for indice in range(len(registro_ventas["producto"])):
    ingreso_total_nativo += (registro_ventas["unidades"][indice]
                             * registro_ventas["precio_unitario"][indice])

# 2. Columna derivada
ingresos_nativos = []
for indice in range(len(registro_ventas["producto"])):
    ingresos_nativos.append(registro_ventas["unidades"][indice]
                            * registro_ventas["precio_unitario"][indice])

# 3. Filtrado por condición
productos_caros_nativo = []
for indice in range(len(registro_ventas["producto"])):
    if registro_ventas["precio_unitario"][indice] > 40:
        productos_caros_nativo.append(registro_ventas["producto"][indice])

# 4. Agregación por grupo: el caso más costoso en Python nativo,
#    porque hay que gestionar a mano la creación de cada clave.
ingreso_por_region_nativo = {}
for indice in range(len(registro_ventas["producto"])):
    region = registro_ventas["region"][indice]
    ingreso = (registro_ventas["unidades"][indice]
               * registro_ventas["precio_unitario"][indice])
    if region in ingreso_por_region_nativo:
        ingreso_por_region_nativo[region] += ingreso
    else:
        ingreso_por_region_nativo[region] = ingreso


# ---------------------------------------------
# VERSIÓN PANDAS
# ---------------------------------------------
# El mismo diccionario, sin modificar una sola línea, es entrada válida.
ventas = pd.DataFrame(registro_ventas)

# paso 2: derivada
ventas["ingreso"] = ventas["unidades"] * ventas["precio_unitario"]
# paso 1: total
ingreso_total = ventas["ingreso"].sum()
# paso 3
productos_caros = ventas.loc[ventas["precio_unitario"] > 40,
                             "producto"].tolist()
# paso 4: agrupación
ingreso_por_region = ventas.groupby("region")["ingreso"].sum()

# ---------------------------------------------
# VERIFICACIÓN DE EQUIVALENCIA
# ---------------------------------------------
print("ingreso total  :", ingreso_total)
print("productos caros:", productos_caros)
print("\ningreso por región:")
print(ingreso_por_region)
print()
print(ingreso_total_nativo == ingreso_total)                     # True
print(ingresos_nativos == ventas["ingreso"].tolist())            # True
print(productos_caros_nativo == productos_caros)                 # True
print(ingreso_por_region_nativo == ingreso_por_region.to_dict())  # True

#### Salida esperada

*Salida en el libro:*

```text
ingreso total  : 2975.0
productos caros: ['Monitor', 'Webcam', 'Monitor']

ingreso por región:
region
Este      860.0
Norte     675.0
Sur      1440.0
Name: ingreso, dtype: float64

True
True
True
True
```

### [Ex-L2-C3-02] Carga defensiva de un CSV sucio

#### Desarrollo con código

In [ ]:
from pathlib import Path

Path("pedidos_sucio.csv").write_text(
    "codigo_cliente|fecha_pedido|region|unidades|importe\n"
    "00123|03/04/2024|Norte|12|1.250,50\n"
    "00456|15/02/2024|Sur|N/D|890,00\n"
    "00789|22/11/2024|Este|8|sin dato\n"
    "01011|05/06/2024|Norte|--|2.100,75\n",
    encoding="utf-8",
)
print(Path("pedidos_sucio.csv").read_text(encoding="utf-8"))

*Salida en el libro:*

```text
codigo_cliente|fecha_pedido|region|unidades|importe
00123|03/04/2024|Norte|12|1.250,50
00456|15/02/2024|Sur|N/D|890,00
00789|22/11/2024|Este|8|sin dato
01011|05/06/2024|Norte|--|2.100,75
```

In [ ]:
import pandas as pd

# ---------------------------------------------
# A) CARGA INGENUA: el resultado es inutilizable
# ---------------------------------------------
ingenua = pd.read_csv("pedidos_sucio.csv")
print(ingenua.shape)      # (4, 1)  -> una sola columna
print(ingenua.columns.tolist())
# ['codigo_cliente|fecha_pedido|region|unidades|importe']

In [ ]:
# ---------------------------------------------
# B) CARGA DEFENSIVA
# ---------------------------------------------
pedidos = pd.read_csv(
    "pedidos_sucio.csv",
    sep="|",                                   # separador real del archivo
    thousands=".",                             # punto como separador de miles
    decimal=",",                               # coma como separador decimal
    encoding="utf-8",
    # las TRES codificaciones de faltante
    na_values=["N/D", "sin dato", "--", ""],
    # preserva los ceros a la izquierda
    dtype={"codigo_cliente": str},
)

# La fecha se convierte APARTE, con el formato declarado explícitamente.
# Es la práctica recomendada en la Sección 3.4; el porqué se ve más abajo,
# en los dos modos de fallo silencioso de parse_dates.
pedidos["fecha_pedido"] = pd.to_datetime(pedidos["fecha_pedido"],
                                         format="%d/%m/%Y")
print(pedidos)
print()
print(pedidos.dtypes)

#### Salida esperada

*Salida en el libro:*

```text
  codigo_cliente fecha_pedido region  unidades  importe
0          00123   2024-04-03  Norte      12.0  1250.50
1          00456   2024-02-15    Sur       NaN   890.00
2          00789   2024-11-22   Este       8.0      NaN
3          01011   2024-06-05  Norte       NaN  2100.75

codigo_cliente            object
fecha_pedido      datetime64[ns]
region                    object
unidades                 float64
importe                  float64
dtype: object
```

In [ ]:
completa = dict(sep="|", thousands=".", decimal=",", encoding="utf-8",
                na_values=["N/D", "sin dato", "--", ""],
                dtype={"codigo_cliente": str})
for omitido in ["thousands", "decimal", "na_values", "dtype"]:
    parametros = {k: v for k, v in completa.items() if k != omitido}
    prueba = pd.read_csv("pedidos_sucio.csv", **parametros)
    print(f"sin {omitido}:")
    print("  tipos       :", prueba.dtypes.astype(str).tolist())
    print("  primera fila:", prueba.iloc[0].astype(str).tolist())

*Salida en el libro:*

```text
sin thousands:
  tipos       : ['object', 'object', 'object', 'float64', 'object']
  primera fila: ['00123', '03/04/2024', 'Norte', '12.0', '1.250,50']
sin decimal:
  tipos       : ['object', 'object', 'object', 'float64', 'object']
  primera fila: ['00123', '03/04/2024', 'Norte', '12.0', '1.250,50']
sin na_values:
  tipos       : ['object', 'object', 'object', 'object', 'object']
  primera fila: ['00123', '03/04/2024', 'Norte', '12', '1.250,50']
sin dtype:
  tipos       : ['int64', 'object', 'object', 'float64', 'float64']
  primera fila: ['123', '03/04/2024', 'Norte', '12.0', '1250.5']
```

#### Los dos modos de fallo silencioso de `parse_dates`

In [ ]:
# MODO 1 — Todas las fechas son ambiguas (día y mes ambos <= 12).
# parse_dates elige el orden estadounidense y convierte MAL, sin avisar.
# Lo correcto sería ['2024-04-03', '2024-06-05', '2024-08-07']: 3 de abril...
Path("todas_ambiguas.csv").write_text(
    "fecha\n03/04/2024\n05/06/2024\n07/08/2024\n", encoding="utf-8")
ambiguas = pd.read_csv("todas_ambiguas.csv", parse_dates=["fecha"])
print("modo 1:", ambiguas["fecha"].dt.strftime("%Y-%m-%d").tolist())

# MODO 2 — La columna MEZCLA fechas ambiguas e inequívocas (como 15/02).
# parse_dates no encuentra un formato consistente, SE RINDE y deja texto.
sin_formato = pd.read_csv("pedidos_sucio.csv", sep="|",
                          parse_dates=["fecha_pedido"])
print("modo 2:", sin_formato["fecha_pedido"].dtype)
print("modo 2:", sin_formato["fecha_pedido"].tolist())

*Salida en el libro:*

```text
modo 1: ['2024-03-04', '2024-05-06', '2024-07-08']
modo 2: object
modo 2: ['03/04/2024', '15/02/2024', '22/11/2024', '05/06/2024']
```

In [ ]:
# La alternativa explícita, en cambio, falla de forma RUIDOSA:
columna_con_formatos_mezclados = pd.Series(["03/04/2024", "2024-05-06"])
try:
    pd.to_datetime(columna_con_formatos_mezclados, format="%d/%m/%Y")
except ValueError as error:
    print(f"ValueError: {error}")

*Salida en el libro:*

```text
ValueError: time data "2024-05-06" doesn't match format "%d/%m/%Y", at position 1. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.
```

### [Ex-L2-C3-03] Auditoría de selecciones `.loc` / `.iloc`

#### Desarrollo con código

In [ ]:
import pandas as pd

catalogo = pd.DataFrame({
    "producto": ["Teclado", "Monitor", "Mouse", "Webcam", "Auriculares"],
    "precio": [25.0, 180.0, 12.5, 45.0, 60.0],
    "stock": [100, 15, 200, 40, 75],
})

# Un filtrado previo conserva el índice ORIGINAL: 0, 2, 3, 4 (falta el 1).
disponibles = catalogo[catalogo["stock"] > 30]
print(disponibles.index.tolist())     # [0, 2, 3, 4]

In [ ]:
try:
    disponibles.loc[1]
except KeyError as error:
    print(f"KeyError: {error}")
# La etiqueta 1 fue eliminada por el filtro; .loc busca ETIQUETAS, no
# posiciones.
print(disponibles.iloc[1]["producto"])   # Mouse -> CORRECTO: la segunda fila

*Salida en el libro:*

```text
KeyError: 1
Mouse
```

In [ ]:
try:
    disponibles.iloc["Teclado"]
except TypeError as error:
    print(f"TypeError: {error}")
print(disponibles.loc[disponibles["producto"] == "Teclado",
                      "precio"].tolist())
# [25.0] -> CORRECTO

*Salida en el libro:*

```text
TypeError: Cannot index by location index with a non-integer key
[25.0]
```

In [ ]:
print(catalogo.loc[0:2, "producto"].tolist())
# ['Teclado', 'Monitor', 'Mouse'] -> INCLUYE la etiqueta 2
print(catalogo.iloc[0:2, 0].tolist())
# ['Teclado', 'Monitor'] -> EXCLUYE la posición 2
# Ambas se ejecutan sin error y devuelven cantidades distintas de filas.

In [ ]:
inventario = catalogo.copy()
inventario[inventario["stock"] < 50]["precio"] = 0.0

*Salida en el libro:*

```text
SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inventario[inventario["stock"] < 50]["precio"] = 0.0
```

In [ ]:
print((inventario["precio"] == 0.0).any())
# False -> la asignación se PERDIÓ

# CORRECTO: una sola indexación con .loc
inventario.loc[inventario["stock"] < 50, "precio"] = 0.0
print((inventario["precio"] == 0.0).sum())
# 2 -> ahora sí cambian las dos filas

In [ ]:
try:
    catalogo[0]    # el acceso directo busca una COLUMNA llamada 0
except KeyError as error:
    print(f"KeyError: {error}")
print(catalogo.iloc[0]["producto"])   # Teclado -> CORRECTO: primera fila
# el acceso directo df[...] es para COLUMNAS
columna = catalogo["producto"]

*Salida en el libro:*

```text
KeyError: 0
Teclado
```

### [Ex-L2-C3-04] Optimización de la huella de memoria

#### Desarrollo con código

In [ ]:
import pandas as pd
import numpy as np

generador = np.random.default_rng(42)
FILAS = 500_000

transacciones = pd.DataFrame({
    "id_transaccion": np.arange(1, FILAS + 1, dtype=np.int64),
    "id_cliente": generador.integers(1, 50_000, FILAS),
    "cantidad": generador.integers(1, 60, FILAS),
    "precio": (generador.random(FILAS) * 800).round(2),
    "region": generador.choice(["Norte", "Sur", "Este", "Oeste"], FILAS),
    "categoria": generador.choice(["Electrónica", "Hogar", "Ropa",
                                   "Deportes"], FILAS),
    "estado": generador.choice(["Completado", "Pendiente", "Cancelado"],
                               FILAS),
})

# deep=True es imprescindible: sin él, Pandas reporta solo el tamaño de los
# punteros de las columnas de texto y subestima el consumo real (Sección 3.9).
memoria_antes = transacciones.memory_usage(deep=True)
print(f"TOTAL: {memoria_antes.sum() / 1024**2:.2f} MB")

# ---------------------------------------------
# 1. JUSTIFICAR EL TIPO DESTINO A PARTIR DEL RANGO REAL
# ---------------------------------------------
TIPOS_CANDIDATOS = [np.uint8, np.uint16, np.uint32, np.uint64]

for columna in ["id_transaccion", "id_cliente", "cantidad"]:
    maximo = transacciones[columna].max()
    # Se elige el primer tipo cuyo máximo representable alcance
    for tipo in TIPOS_CANDIDATOS:
        if maximo <= np.iinfo(tipo).max:
            break
    print(f"{columna}: max={maximo} -> {np.dtype(tipo).name}")

# Cardinalidad: 'category' solo compensa si los valores únicos son POCOS
for columna in ["region", "categoria", "estado"]:
    unicos = transacciones[columna].nunique()
    print(f"{columna}: {unicos} únicos de {FILAS} filas")

# ---------------------------------------------
# 2. APLICAR LA CONVERSIÓN
# ---------------------------------------------
optimizado = transacciones.copy()
optimizado["id_transaccion"] = optimizado["id_transaccion"].astype("uint32")
optimizado["id_cliente"] = optimizado["id_cliente"].astype("uint16")
optimizado["cantidad"] = optimizado["cantidad"].astype("uint8")
optimizado["precio"] = optimizado["precio"].astype("float32")
for columna in ["region", "categoria", "estado"]:
    optimizado[columna] = optimizado[columna].astype("category")

memoria_despues = optimizado.memory_usage(deep=True)
print(f"\n{'Columna':<16}{'Tipo antes':>11}{'MB antes':>10}"
      f"{'Tipo después':>14}{'MB después':>12}")
for columna in transacciones.columns:
    print(f"{columna:<16}{str(transacciones[columna].dtype):>11}"
          f"{memoria_antes[columna] / 1024**2:>10.3f}"
          f"{str(optimizado[columna].dtype):>14}"
          f"{memoria_despues[columna] / 1024**2:>12.3f}")
total_antes = memoria_antes.sum() / 1024**2
total_despues = memoria_despues.sum() / 1024**2
print(f"{'TOTAL':<16}{total_antes:>21.2f}{total_despues:>26.2f}")
print(f"ahorro: {1 - total_despues / total_antes:.1%}")

# ---------------------------------------------
# 3. VERIFICAR QUE NADA SE CORROMPIÓ
# ---------------------------------------------
for columna in ["id_transaccion", "id_cliente", "cantidad", "region",
                "categoria", "estado"]:
    print(columna, (transacciones[columna] == optimizado[columna]).all())

# ---------------------------------------------
# 4. CUANTIFICAR LA PRECISIÓN PERDIDA
# ---------------------------------------------
diferencia = (transacciones["precio"]
              - optimizado["precio"].astype("float64")).abs()
print(f"error absoluto máximo: {diferencia.max():.6f}")
print(f"error relativo máximo: "
      f"{(diferencia / transacciones['precio']).max()*100:.6f}%")

#### Salida esperada

*Salida en el libro:*

```text
TOTAL: 97.08 MB
id_transaccion: max=500000 -> uint32
id_cliente: max=49999 -> uint16
cantidad: max=59 -> uint8
region: 4 únicos de 500000 filas
categoria: 4 únicos de 500000 filas
estado: 3 únicos de 500000 filas

Columna          Tipo antes  MB antes  Tipo después  MB después
id_transaccion        int64     3.815        uint32       1.907
id_cliente            int64     3.815        uint16       0.954
cantidad              int64     3.815         uint8       0.477
precio              float64     3.815       float32       1.907
region               object    25.391      category       0.477
categoria            object    28.610      category       0.477
estado               object    27.816      category       0.477
TOTAL                           97.08                      6.68
ahorro: 93.1%
id_transaccion True
id_cliente True
cantidad True
region True
categoria True
estado True
error absoluto máximo: 0.000029
error relativo máximo: 0.000006%
```

#### Puntos de aprendizaje

In [ ]:
prueba = pd.Series([10, 300, 50])
pd.to_numeric(prueba, downcast="unsigned").tolist()
# [10, 300, 50]  -> eligió uint16
prueba.astype("uint8").tolist()
# [10, 44, 50]  -> ¡300 pasó a 44!